# M08 vs M09 evaluation

This notebook evaluates whether replacing the global average pooling of M08 with the M09 multi-slot temporal attention pooling improves CBC parameter regression.

Comparison:

- **M08**: residual dilated encoder + `AdaptiveAvgPool1d(1)`
- **M09**: same residual dilated encoder + multi-slot temporal attention pooling

Use the same 500k dataset, same split, same seed, same loss, same optimizer settings, and same evaluation splits.

Primary question:

> Does M09 improve point-estimation accuracy and error tails relative to M08 enough to justify the more complex pooling mechanism?


## 1. Imports and paths

In [ ]:
from pathlib import Path
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

DATA_ROOT = Path("/data/vserrano/cbc_pe_data")
RESULTS_ROOT = DATA_ROOT / "results"
CHECKPOINT_ROOT = DATA_ROOT / "models" / "checkpoints"
PROCESSED_ROOT = DATA_ROOT / "processed"

DATASET_ID = "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000"
DATASET_RESULTS_ROOT = RESULTS_ROOT / DATASET_ID
DATASET_CHECKPOINT_ROOT = CHECKPOINT_ROOT / DATASET_ID
PROCESSED_ROOT = PROCESSED_ROOT / DATASET_ID

PROJECT_ROOT = Path("/afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe/")
if PROJECT_ROOT.exists() and str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

LABEL_NAMES = ["chirp_mass", "total_mass", "chi_eff"]
SPLITS = ["val", "cal", "test"]

print("DATASET_RESULTS_ROOT:", DATASET_RESULTS_ROOT, DATASET_RESULTS_ROOT.exists())
print("DATASET_CHECKPOINT_ROOT:", DATASET_CHECKPOINT_ROOT, DATASET_CHECKPOINT_ROOT.exists())
print("PROJECT_ROOT:", PROJECT_ROOT, PROJECT_ROOT.exists())

## 2. Resolve files

Review the selected files before continuing. The filename patterns are flexible to tolerate small naming differences.

In [ ]:
def resolve_file(root: Path, patterns, description: str) -> Path:
    if isinstance(patterns, (str, Path)):
        patterns = [patterns]

    matches = []
    for pattern in patterns:
        matches.extend(root.glob(str(pattern)))

    matches = sorted(set(matches), key=lambda p: p.stat().st_mtime)

    if not matches:
        raise FileNotFoundError(
            f"Could not find {description} in {root}.\n"
            f"Patterns tried: {[str(p) for p in patterns]}"
        )

    print(f"\n{description}:")
    for path in matches:
        print("  ", path.name)

    selected = matches[-1]
    print("Selected:", selected.name)
    return selected


RUNS = {
    "M08": {
        "label": "M08 avg pooling",
        "prediction_patterns": [
            f"{DATASET_ID}_SimpleCNN_ResidualDilated_batchslices_bs256_MSELoss_seed123_val_cal_test_predictions_embeddings.npz",


        ],
        "history_patterns": [
            f"{DATASET_ID}_SimpleCNN_ResidualDilated_batchslices_bs256_MSELoss_seed123_history.npz",

        ],
        "checkpoint_patterns": [
            f"{DATASET_ID}_SimpleCNN_ResidualDilated_batchslices_bs256_MSELoss_seed123_checkpoint.pt",

        ],
    },
    "M09": {
        "label": "M09 multi-slot attention pooling",
        "prediction_patterns": [
            f"{DATASET_ID}_SimpleCNN_ResidualDilatedMultiAttention_M09_multiattn_k4_sd32_emb64_d124_bs256_MSELoss_seed123_val_cal_test_predictions_embeddings.npz",
            f"{DATASET_ID}_SimpleCNN_ResidualDilatedMultiAttention*M09*seed123*val_cal_test*predictions_embeddings.npz",
            f"{DATASET_ID}_SimpleCNN_ResidualDilatedMultiAttention*seed123*predictions_embeddings.npz",
        ],
        "history_patterns": [
            f"{DATASET_ID}_SimpleCNN_ResidualDilatedMultiAttention_M09_multiattn_k4_sd32_emb64_d124_bs256_MSELoss_seed123_history.npz",
            f"{DATASET_ID}_SimpleCNN_ResidualDilatedMultiAttention*M09*seed123*history.npz",
            f"{DATASET_ID}_SimpleCNN_ResidualDilatedMultiAttention*seed123*history.npz",
        ],
        "checkpoint_patterns": [
            f"{DATASET_ID}_SimpleCNN_ResidualDilatedMultiAttention_M09_multiattn_k4_sd32_emb64_d124_bs256_MSELoss_seed123_checkpoint.pt",
            f"{DATASET_ID}_SimpleCNN_ResidualDilatedMultiAttention*M09*seed123*checkpoint.pt",
            f"{DATASET_ID}_SimpleCNN_ResidualDilatedMultiAttention*seed123*checkpoint.pt",
        ],
    },
}

for run_id, run in RUNS.items():
    run["prediction_path"] = resolve_file(DATASET_RESULTS_ROOT, run["prediction_patterns"], f"{run_id} prediction file")
    run["history_path"] = resolve_file(DATASET_RESULTS_ROOT, run["history_patterns"], f"{run_id} history file")
    run["checkpoint_path"] = resolve_file(DATASET_CHECKPOINT_ROOT, run["checkpoint_patterns"], f"{run_id} checkpoint")

for run_id, run in RUNS.items():
    print(run_id, "prediction:", run["prediction_path"].name)
    print(run_id, "history:   ", run["history_path"].name)
    print(run_id, "checkpoint:", run["checkpoint_path"].name)

In [ ]:
def read_hdf5_dataset_preserve_order(
    h5_file,
    dataset_key,
    indices,
    dtype=None,
):
    """
    Read an HDF5 dataset with arbitrary indices while preserving the
    original order requested by the caller.

    h5py requires fancy indices to be sorted increasingly. This helper
    sorts indices for reading, then restores the original order.

    Parameters
    ----------
    h5_file : h5py.File
        Open HDF5 file.

    dataset_key : str
        Dataset path inside the HDF5 file, e.g. "X" or "snr/network_snr".

    indices : array-like
        Event indices in the desired output order.

    dtype : numpy dtype, optional
        Optional dtype conversion.

    Returns
    -------
    array : np.ndarray
        Data in the same order as `indices`.
    """
    indices = np.asarray(indices, dtype=np.int64)

    if indices.ndim != 1:
        raise ValueError(
            f"indices must be 1D, got shape {indices.shape}"
        )

    # Stable sort is safer if duplicated indices ever appear.
    order = np.argsort(indices, kind="stable")
    sorted_indices = indices[order]

    data_sorted = np.asarray(
        h5_file[dataset_key][sorted_indices],
        dtype=dtype,
    )

    # inverse_order maps sorted read order back to original requested order.
    inverse_order = np.empty_like(order)
    inverse_order[order] = np.arange(len(order))

    return data_sorted[inverse_order]

## 3. Load label statistics and prediction files

In [ ]:
LABEL_STATS_PATH = resolve_file(
    PROCESSED_ROOT ,
    [
        f"{DATASET_ID}_label_stats_train_only_train400000_val40000_cal30000_test30000_seed123.npz",
        f"{DATASET_ID}*label_stats*seed123.npz",
    ],
    "label statistics",
)

with np.load(LABEL_STATS_PATH) as stats:
    y_mean = np.asarray(stats["y_mean"], dtype=float)
    y_std = np.asarray(stats["y_std"], dtype=float)

print("y_mean:", y_mean)
print("y_std:", y_std)


def load_prediction_file(path: Path):
    out = {}
    with np.load(path, allow_pickle=True) as data:
        print(f"\n{path.name}")
        for key in sorted(data.files):
            arr = np.asarray(data[key])
            print(f"  {key:24s} shape={str(arr.shape):16s} dtype={arr.dtype}")
            out[key] = arr
    return out


RAW = {run_id: load_prediction_file(run["prediction_path"]) for run_id, run in RUNS.items()}

## 4. Align splits by HDF5 indices

In [ ]:
def align_runs_for_split(raw_by_run, split, reference_run="M08"):
    required = [f"idx_{split}", f"pred_{split}", f"y_{split}", f"emb_{split}"]

    for run_id, data in raw_by_run.items():
        missing = [key for key in required if key not in data]
        if missing:
            raise KeyError(f"{run_id} is missing keys for {split}: {missing}")

    common_idx = None
    for data in raw_by_run.values():
        idx = np.asarray(data[f"idx_{split}"], dtype=np.int64)
        common_idx = idx if common_idx is None else np.intersect1d(common_idx, idx)

    common_idx = np.sort(common_idx)
    aligned = {}

    for run_id, data in raw_by_run.items():
        idx = np.asarray(data[f"idx_{split}"], dtype=np.int64)
        order = np.argsort(idx)
        idx_sorted = idx[order]

        positions = np.searchsorted(idx_sorted, common_idx)
        if not np.array_equal(idx_sorted[positions], common_idx):
            raise RuntimeError(f"Alignment failure for {run_id}, split={split}")

        aligned[run_id] = {
            "idx": common_idx,
            "pred": np.asarray(data[f"pred_{split}"])[order][positions],
            "y": np.asarray(data[f"y_{split}"])[order][positions],
            "emb": np.asarray(data[f"emb_{split}"])[order][positions],
        }

    y_ref = aligned[reference_run]["y"]
    for run_id in aligned:
        if not np.allclose(aligned[run_id]["y"], y_ref, rtol=1e-5, atol=1e-5):
            raise AssertionError(f"Targets do not agree after alignment: {reference_run} vs {run_id}")

    return aligned


ALIGNED = {}
for split in SPLITS:
    ALIGNED[split] = align_runs_for_split(RAW, split)
    print(f"{split:5s}: {len(ALIGNED[split]['M08']['idx']):,} common samples")

## 5. Regression metrics

In [ ]:
def inverse_standardize(y):
    return y * y_std + y_mean


def regression_metrics(y_true, y_pred, run_id, split, space):
    residual = y_pred - y_true
    abs_error = np.abs(residual)

    mse = np.mean(residual**2, axis=0)
    rmse = np.sqrt(mse)
    mae = np.mean(abs_error, axis=0)
    bias = np.mean(residual, axis=0)
    residual_std = np.std(residual, axis=0)
    median_ae = np.median(abs_error, axis=0)

    ss_res = np.sum(residual**2, axis=0)
    ss_tot = np.sum((y_true - np.mean(y_true, axis=0, keepdims=True)) ** 2, axis=0)
    r2 = 1.0 - ss_res / ss_tot

    rows = []
    for j, label in enumerate(LABEL_NAMES):
        rows.append({
            "run_id": run_id,
            "split": split,
            "space": space,
            "label": label,
            "MSE": float(mse[j]),
            "RMSE": float(rmse[j]),
            "MAE": float(mae[j]),
            "bias": float(bias[j]),
            "median_abs_error": float(median_ae[j]),
            "residual_std": float(residual_std[j]),
            "R2": float(r2[j]),
        })
    return rows


metric_rows = []

for split in SPLITS:
    y_true_std = ALIGNED[split]["M08"]["y"]
    y_true_phys = inverse_standardize(y_true_std)

    for run_id in RUNS:
        pred_std = ALIGNED[split][run_id]["pred"]
        pred_phys = inverse_standardize(pred_std)

        metric_rows.extend(regression_metrics(y_true_std, pred_std, run_id, split, "standardized"))
        metric_rows.extend(regression_metrics(y_true_phys, pred_phys, run_id, split, "physical"))

metrics_df = pd.DataFrame(metric_rows)

global_metrics_df = (
    metrics_df
    .groupby(["run_id", "split", "space"], as_index=False)
    .agg(
        global_MSE=("MSE", "mean"),
        global_RMSE=("RMSE", "mean"),
        global_MAE=("MAE", "mean"),
        mean_R2=("R2", "mean"),
    )
)

display(global_metrics_df.query("space == 'standardized'"))
display(metrics_df.query("split == 'test' and space == 'standardized'").sort_values(["label", "run_id"]))

## 6. M09 vs M08 differences

For error metrics, negative values mean M09 improves over M08.

In [ ]:
def difference_table(metrics, split="test", space="standardized"):
    sub = metrics.query("split == @split and space == @space")
    pivot = sub.pivot(index="label", columns="run_id")

    rows = []
    for label in LABEL_NAMES:
        row = {"label": label}

        for metric in ["MSE", "RMSE", "MAE"]:
            m08 = float(pivot.loc[label, (metric, "M08")])
            m09 = float(pivot.loc[label, (metric, "M09")])
            row[f"{metric}_M08"] = m08
            row[f"{metric}_M09"] = m09
            row[f"{metric}_delta_pct"] = 100 * (m09 - m08) / m08

        for metric in ["R2", "bias"]:
            m08 = float(pivot.loc[label, (metric, "M08")])
            m09 = float(pivot.loc[label, (metric, "M09")])
            row[f"{metric}_M08"] = m08
            row[f"{metric}_M09"] = m09
            row[f"{metric}_delta"] = m09 - m08

        rows.append(row)

    return pd.DataFrame(rows)


test_difference_df = difference_table(metrics_df, split="test", space="standardized")
display(test_difference_df)

global_test = global_metrics_df.query("split == 'test' and space == 'standardized'").set_index("run_id")

global_difference_df = pd.DataFrame([{
    "global_MSE_M08": global_test.loc["M08", "global_MSE"],
    "global_MSE_M09": global_test.loc["M09", "global_MSE"],
    "global_MSE_delta_pct": 100 * (global_test.loc["M09", "global_MSE"] - global_test.loc["M08", "global_MSE"]) / global_test.loc["M08", "global_MSE"],
    "global_MAE_M08": global_test.loc["M08", "global_MAE"],
    "global_MAE_M09": global_test.loc["M09", "global_MAE"],
    "global_MAE_delta_pct": 100 * (global_test.loc["M09", "global_MAE"] - global_test.loc["M08", "global_MAE"]) / global_test.loc["M08", "global_MAE"],
}])

display(global_difference_df)

## 7. Regression-to-the-mean diagnostics

In [ ]:
def slope_diagnostics(y_true, y_pred, run_id, split):
    rows = []

    for j, label in enumerate(LABEL_NAMES):
        true = y_true[:, j]
        pred = y_pred[:, j]

        slope, intercept = np.polyfit(true, pred, deg=1)
        corr = np.corrcoef(true, pred)[0, 1]

        q01_true, q99_true = np.quantile(true, [0.01, 0.99])
        q01_pred, q99_pred = np.quantile(pred, [0.01, 0.99])

        rows.append({
            "run_id": run_id,
            "split": split,
            "label": label,
            "slope": float(slope),
            "intercept": float(intercept),
            "correlation": float(corr),
            "std_ratio_pred_over_true": float(np.std(pred) / np.std(true)),
            "q01_q99_ratio_pred_over_true": float((q99_pred - q01_pred) / (q99_true - q01_true)),
        })

    return rows


slope_rows = []

for split in SPLITS:
    y_true = ALIGNED[split]["M08"]["y"]

    for run_id in RUNS:
        slope_rows.extend(slope_diagnostics(y_true, ALIGNED[split][run_id]["pred"], run_id, split))

slope_df = pd.DataFrame(slope_rows)

display(slope_df.query("split == 'test'").sort_values(["label", "run_id"]))

slope_pivot = slope_df.query("split == 'test'").pivot(index="label", columns="run_id")

slope_delta_rows = []
for label in LABEL_NAMES:
    slope_delta_rows.append({
        "label": label,
        "slope_M08": slope_pivot.loc[label, ("slope", "M08")],
        "slope_M09": slope_pivot.loc[label, ("slope", "M09")],
        "slope_delta": slope_pivot.loc[label, ("slope", "M09")] - slope_pivot.loc[label, ("slope", "M08")],
        "std_ratio_M08": slope_pivot.loc[label, ("std_ratio_pred_over_true", "M08")],
        "std_ratio_M09": slope_pivot.loc[label, ("std_ratio_pred_over_true", "M09")],
        "std_ratio_delta": slope_pivot.loc[label, ("std_ratio_pred_over_true", "M09")] - slope_pivot.loc[label, ("std_ratio_pred_over_true", "M08")],
    })

slope_delta_df = pd.DataFrame(slope_delta_rows)
display(slope_delta_df)

In [ ]:
for label in LABEL_NAMES:
    sub = slope_df.query("split == 'test' and label == @label").set_index("run_id").loc[["M08", "M09"]]

    plt.figure(figsize=(7.0, 4.3))
    plt.plot(sub.index, sub["slope"], marker="o", label="slope")
    plt.plot(sub.index, sub["std_ratio_pred_over_true"], marker="s", label="std ratio")
    plt.axhline(1.0, linestyle="--", linewidth=1)
    plt.ylabel("Ratio / slope")
    plt.title(f"M08 vs M09 contraction — {label}")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 8. Physical absolute-error quantiles

In [ ]:
QUANTILES = [0.50, 0.90, 0.95, 0.99]
quantile_rows = []

for split in SPLITS:
    y_true_phys = inverse_standardize(ALIGNED[split]["M08"]["y"])

    for run_id in RUNS:
        pred_phys = inverse_standardize(ALIGNED[split][run_id]["pred"])
        ae = np.abs(pred_phys - y_true_phys)

        for j, label in enumerate(LABEL_NAMES):
            for q in QUANTILES:
                quantile_rows.append({
                    "run_id": run_id,
                    "split": split,
                    "label": label,
                    "quantile": q,
                    "absolute_error": float(np.quantile(ae[:, j], q)),
                })

quantiles_df = pd.DataFrame(quantile_rows)

test_quantile_df = (
    quantiles_df.query("split == 'test'")
    .pivot_table(index=["label", "quantile"], columns="run_id", values="absolute_error")
    .reset_index()
)

test_quantile_df["M09_vs_M08_pct"] = 100 * (test_quantile_df["M09"] - test_quantile_df["M08"]) / test_quantile_df["M08"]

display(test_quantile_df)

## 9. Paired bootstrap on test-set metric deltas

In [ ]:
def paired_bootstrap_metric_delta(y_true, pred_a, pred_b, metric="mse", n_boot=2000, seed=123):
    rng = np.random.default_rng(seed)
    n = y_true.shape[0]
    n_labels = y_true.shape[1]

    deltas = np.empty((n_boot, n_labels), dtype=float)

    err_a = pred_a - y_true
    err_b = pred_b - y_true

    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)

        if metric == "mse":
            va = np.mean(err_a[idx] ** 2, axis=0)
            vb = np.mean(err_b[idx] ** 2, axis=0)
        elif metric == "mae":
            va = np.mean(np.abs(err_a[idx]), axis=0)
            vb = np.mean(np.abs(err_b[idx]), axis=0)
        else:
            raise ValueError(metric)

        deltas[i] = vb - va

    return deltas


y_true_test = ALIGNED["test"]["M08"]["y"]
pred_m08_test = ALIGNED["test"]["M08"]["pred"]
pred_m09_test = ALIGNED["test"]["M09"]["pred"]

bootstrap_rows = []

for metric in ["mse", "mae"]:
    deltas = paired_bootstrap_metric_delta(y_true_test, pred_m08_test, pred_m09_test, metric=metric, n_boot=2000, seed=123)

    if metric == "mse":
        point = np.mean((pred_m09_test - y_true_test) ** 2, axis=0) - np.mean((pred_m08_test - y_true_test) ** 2, axis=0)
    else:
        point = np.mean(np.abs(pred_m09_test - y_true_test), axis=0) - np.mean(np.abs(pred_m08_test - y_true_test), axis=0)

    for j, label in enumerate(LABEL_NAMES):
        bootstrap_rows.append({
            "metric": metric.upper(),
            "label": label,
            "delta_M09_minus_M08": float(point[j]),
            "ci95_low": float(np.quantile(deltas[:, j], 0.025)),
            "ci95_high": float(np.quantile(deltas[:, j], 0.975)),
            "prob_delta_below_zero": float(np.mean(deltas[:, j] < 0)),
        })

bootstrap_df = pd.DataFrame(bootstrap_rows)
display(bootstrap_df)

## 10. Per-event win rates and disagreement

In [ ]:
ae_m08 = np.abs(pred_m08_test - y_true_test)
ae_m09 = np.abs(pred_m09_test - y_true_test)

win_rows = []
for j, label in enumerate(LABEL_NAMES):
    disagreement = pred_m09_test[:, j] - pred_m08_test[:, j]

    win_rows.append({
        "label": label,
        "M08_win_rate": float(np.mean(ae_m08[:, j] < ae_m09[:, j])),
        "M09_win_rate": float(np.mean(ae_m09[:, j] < ae_m08[:, j])),
        "tie_rate": float(np.mean(ae_m08[:, j] == ae_m09[:, j])),
        "prediction_difference_mean": float(np.mean(disagreement)),
        "prediction_difference_std": float(np.std(disagreement)),
        "prediction_difference_q95_abs": float(np.quantile(np.abs(disagreement), 0.95)),
    })

win_rate_df = pd.DataFrame(win_rows)
display(win_rate_df)

## 11. Training histories and checkpoint metadata

In [ ]:
def load_history(path):
    with np.load(path, allow_pickle=True) as data:
        out = {key: np.asarray(data[key]) for key in data.files}
    print(path.name, sorted(out))
    return out


def first_existing(mapping, candidates):
    for key in candidates:
        if key in mapping:
            return np.asarray(mapping[key])
    return None


HISTORIES = {run_id: load_history(run["history_path"]) for run_id, run in RUNS.items()}

history_rows = []

plt.figure(figsize=(8.5, 5.0))

for run_id, history in HISTORIES.items():
    train_loss = first_existing(history, ["train_loss", "train_losses", "loss_train"])
    val_loss = first_existing(history, ["val_loss", "val_losses", "loss_val"])

    if train_loss is None or val_loss is None:
        print(f"Missing loss keys for {run_id}")
        continue

    best_idx = int(np.nanargmin(val_loss))
    best_epoch = best_idx + 1
    stop_epoch = len(val_loss)

    history_rows.append({
        "run_id": run_id,
        "best_epoch": best_epoch,
        "stop_epoch": stop_epoch,
        "best_val_loss": float(val_loss[best_idx]),
        "final_train_loss": float(train_loss[-1]),
        "final_val_loss": float(val_loss[-1]),
    })

    epochs = np.arange(1, len(val_loss) + 1)
    plt.plot(epochs, val_loss, label=run_id)

plt.xlabel("Epoch")
plt.ylabel("Validation MSE")
plt.title("Validation-loss comparison")
plt.legend()
plt.tight_layout()
plt.show()

history_summary_df = pd.DataFrame(history_rows)
display(history_summary_df)

In [ ]:
checkpoint_rows = []

try:
    import torch

    for run_id, run in RUNS.items():
        checkpoint = torch.load(run["checkpoint_path"], map_location="cpu")

        model_config = checkpoint.get("model_config", {})
        model_kwargs = (
            model_config.get("kwargs")
            or model_config.get("model_kwargs")
            or checkpoint.get("model_kwargs")
            or {}
        )

        checkpoint_rows.append({
            "run_id": run_id,
            "checkpoint_name": run["checkpoint_path"].name,
            "saved_epoch": checkpoint.get("epoch", np.nan),
            "best_val_loss": checkpoint.get("best_val_loss", np.nan),
            "architecture": model_config.get("architecture"),
            "class_name": model_config.get("class_name"),
            "training_seed": model_config.get("training_seed", checkpoint.get("training_seed")),
            "split_seed": model_config.get("split_seed", checkpoint.get("split_seed")),
            "dilations": model_kwargs.get("dilations"),
            "embedding_dim": model_kwargs.get("embedding_dim"),
            "attention_slots": model_kwargs.get("attention_slots"),
            "attention_hidden": model_kwargs.get("attention_hidden"),
            "attention_dropout": model_kwargs.get("attention_dropout"),
            "slot_dim": model_kwargs.get("slot_dim"),
        })

    checkpoint_df = pd.DataFrame(checkpoint_rows)
    display(checkpoint_df)

except Exception as exc:
    print("Checkpoint inspection skipped or failed:")
    print(type(exc).__name__, exc)
    checkpoint_df = pd.DataFrame()

## 12. Optional attention-map diagnostic

This section requires access to the HDF5 dataset and the M09 checkpoint/model class. It is disabled by default. Enable it only after the metric comparison.

In [ ]:
RUN_ATTENTION_DIAGNOSTIC = False

if RUN_ATTENTION_DIAGNOSTIC:
    import h5py
    import torch
    from src.models.network import SimpleCNN_ResidualDilatedMultiAttention

    dataset_path = PROCESSED_ROOT / f"{DATASET_ID}.h5"
    assert dataset_path.exists(), dataset_path

    m09_checkpoint = torch.load(RUNS["M09"]["checkpoint_path"], map_location="cpu")
    m09_model_config = m09_checkpoint.get("model_config", {})
    m09_kwargs = m09_model_config.get("model_kwargs", {})

    # Fallbacks if checkpoint metadata is incomplete.
    m09_kwargs.setdefault("n_detectors", 3)
    m09_kwargs.setdefault("n_outputs", 3)
    m09_kwargs.setdefault("embedding_dim", 64)
    m09_kwargs.setdefault("residual_channels", 64)
    m09_kwargs.setdefault("dilations", [1, 2, 4])
    m09_kwargs.setdefault("residual_kernel_size", 7)
    m09_kwargs.setdefault("dropout_conv", 0.05)
    m09_kwargs.setdefault("dropout_dense", 0.1)
    m09_kwargs.setdefault("num_groups", 8)
    m09_kwargs.setdefault("attention_slots", 4)
    m09_kwargs.setdefault("attention_hidden", 64)
    m09_kwargs.setdefault("attention_dropout", 0.0)
    m09_kwargs.setdefault("slot_dim", 32)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = SimpleCNN_ResidualDilatedMultiAttention(**m09_kwargs).to(device)
    model.load_state_dict(m09_checkpoint["model_state_dict"])
    model.eval()

    idx_test = ALIGNED["test"]["M09"]["idx"]
    rng = np.random.default_rng(123)
    selected_positions = rng.choice(len(idx_test), size=6, replace=False)
    selected_hdf5_idx = idx_test[selected_positions]

    with h5py.File(dataset_path, "r") as f:
        X = np.asarray(f["X"][np.sort(selected_hdf5_idx)], dtype=np.float32)

    x = torch.from_numpy(X).to(device)

    with torch.no_grad():
        features, attn = model.get_attention_weights(x)
        attn = attn.detach().cpu().numpy()

    for i, sample_idx in enumerate(selected_hdf5_idx):
        plt.figure(figsize=(8.5, 4.0))
        for k in range(attn.shape[1]):
            plt.plot(attn[i, k], label=f"slot {k}")
        plt.xlabel("Encoder time index")
        plt.ylabel("Attention weight")
        plt.title(f"M09 attention windows — HDF5 idx {sample_idx}")
        plt.legend()
        plt.tight_layout()
        plt.show()

else:
    print("Attention diagnostic disabled. Set RUN_ATTENTION_DIAGNOSTIC = False to run it.")

## 13. Decision table

In [ ]:
decision_rows = []

for label in LABEL_NAMES:
    metric_row = test_difference_df.set_index("label").loc[label]
    slope_row = slope_delta_df.set_index("label").loc[label]

    q95 = test_quantile_df.query("label == @label and quantile == 0.95").iloc[0]
    q99 = test_quantile_df.query("label == @label and quantile == 0.99").iloc[0]

    boot_mse = bootstrap_df.query("label == @label and metric == 'MSE'").iloc[0]
    boot_mae = bootstrap_df.query("label == @label and metric == 'MAE'").iloc[0]

    decision_rows.append({
        "label": label,
        "MSE_delta_pct": metric_row["MSE_delta_pct"],
        "MAE_delta_pct": metric_row["MAE_delta_pct"],
        "R2_delta": metric_row["R2_delta"],
        "slope_delta": slope_row["slope_delta"],
        "std_ratio_delta": slope_row["std_ratio_delta"],
        "q95_abs_error_delta_pct": q95["M09_vs_M08_pct"],
        "q99_abs_error_delta_pct": q99["M09_vs_M08_pct"],
        "M09_win_rate": win_rate_df.set_index("label").loc[label, "M09_win_rate"],
        "bootstrap_MSE_ci95_low": boot_mse["ci95_low"],
        "bootstrap_MSE_ci95_high": boot_mse["ci95_high"],
        "bootstrap_MAE_ci95_low": boot_mae["ci95_low"],
        "bootstrap_MAE_ci95_high": boot_mae["ci95_high"],
    })

decision_df = pd.DataFrame(decision_rows)

display(global_difference_df)
display(decision_df)

## 14. Save tables

In [ ]:
OUTPUT_DIR = RESULTS_ROOT / "evaluation_M08_vs_M09"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tables = {
    "metrics_all.csv": metrics_df,
    "global_metrics.csv": global_metrics_df,
    "test_differences.csv": test_difference_df,
    "global_test_difference.csv": global_difference_df,
    "slope_diagnostics.csv": slope_df,
    "slope_deltas.csv": slope_delta_df,
    "absolute_error_quantiles.csv": quantiles_df,
    "test_quantile_differences.csv": test_quantile_df,
    "paired_bootstrap.csv": bootstrap_df,
    "win_rates_and_disagreement.csv": win_rate_df,
    "training_summary.csv": history_summary_df,
    "decision_table.csv": decision_df,
}

if not checkpoint_df.empty:
    tables["checkpoint_metadata.csv"] = checkpoint_df

for filename, dataframe in tables.items():
    path = OUTPUT_DIR / filename
    dataframe.to_csv(path, index=False)
    print("Saved:", path)

## 15. True-vs-predicted density plots

These plots show the test-set density in physical units. The diagonal corresponds to perfect prediction.

Use them to inspect:

- regression to the mean;
- compression at the high/low ends of the parameter range;
- whether M09 changes the shape of the prediction cloud relative to M08;
- whether small metric gains correspond to visible structural improvements.


In [ ]:
def plot_true_pred_density(
    y_true,
    y_pred,
    label,
    model_name,
    gridsize=80,
    mincnt=1,
):
    j = LABEL_NAMES.index(label)

    true = y_true[:, j]
    pred = y_pred[:, j]

    lo = min(np.min(true), np.min(pred))
    hi = max(np.max(true), np.max(pred))
    pad = 0.03 * (hi - lo)
    lo -= pad
    hi += pad

    plt.figure(figsize=(6.0, 5.4))
    hb = plt.hexbin(
        true,
        pred,
        gridsize=gridsize,
        mincnt=mincnt,
        bins="log",
    )
    plt.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1)
    plt.xlim(lo, hi)
    plt.ylim(lo, hi)
    plt.xlabel(f"True {label}")
    plt.ylabel(f"Predicted {label}")
    plt.title(f"{model_name} — true vs predicted density — {label}")
    plt.colorbar(hb, label="log10(count)")
    plt.tight_layout()
    plt.show()


y_true_test_phys = inverse_standardize(ALIGNED["test"]["M08"]["y"])

for label in LABEL_NAMES:
    for run_id in ["M08", "M09"]:
        y_pred_test_phys = inverse_standardize(ALIGNED["test"][run_id]["pred"])
        plot_true_pred_density(
            y_true=y_true_test_phys,
            y_pred=y_pred_test_phys,
            label=label,
            model_name=run_id,
            gridsize=85,
            mincnt=1,
        )


## 16. Residual density plots: pred - true vs true

In [ ]:


def plot_residual_density_by_true(
    run_ids=("M08", "M09"),
    split="test",
    physical_space=True,
    gridsize=80,
    trend_bins=35,
    mincnt=1,
):
    """
    Plot residual density:
        residual = prediction - truth
    against true physical/standardized target value.

    Interpretation
    --------------
    - y = 0 means no bias.
    - residual > 0 means overprediction.
    - residual < 0 means underprediction.
    - A sloped median residual curve indicates regression-to-the-mean.
    """

    for label_idx, label in enumerate(LABEL_NAMES):
        fig, axes = plt.subplots(
            1,
            len(run_ids),
            figsize=(6.5 * len(run_ids), 5.2),
            sharey=True,
        )

        if len(run_ids) == 1:
            axes = [axes]

        for ax, run_id in zip(axes, run_ids):
            y_true_std = ALIGNED[split][run_id]["y"][:, label_idx]
            y_pred_std = ALIGNED[split][run_id]["pred"][:, label_idx]

            if physical_space:
                y_true = y_true_std * y_std[label_idx] + y_mean[label_idx]
                y_pred = y_pred_std * y_std[label_idx] + y_mean[label_idx]
                unit = "physical"
            else:
                y_true = y_true_std
                y_pred = y_pred_std
                unit = "standardized"

            residual = y_pred - y_true

            hb = ax.hexbin(
                y_true,
                residual,
                gridsize=gridsize,
                bins="log",
                mincnt=mincnt,
            )

            fig.colorbar(
                hb,
                ax=ax,
                label="log10(count)",
            )

            ax.axhline(
                0.0,
                linestyle="--",
                linewidth=1.2,
            )

            # Median residual trend in true-value bins.
            edges = np.quantile(
                y_true,
                np.linspace(0.0, 1.0, trend_bins + 1),
            )
            edges = np.unique(edges)

            centers = []
            medians = []
            q16 = []
            q84 = []

            for left, right in zip(edges[:-1], edges[1:]):
                mask = (y_true >= left) & (y_true <= right)
                if mask.sum() < 20:
                    continue

                centers.append(np.median(y_true[mask]))
                medians.append(np.median(residual[mask]))
                q16.append(np.quantile(residual[mask], 0.16))
                q84.append(np.quantile(residual[mask], 0.84))

            centers = np.asarray(centers)
            medians = np.asarray(medians)
            q16 = np.asarray(q16)
            q84 = np.asarray(q84)

            ax.plot(
                centers,
                medians,
                linewidth=2.0,
                label="median residual",
                color="red"
            )

            ax.fill_between(
                centers,
                q16,
                q84,
                alpha=0.20,
                label="16–84%",

            )

            ax.set_xlabel(f"True {label} [{unit}]")
            ax.set_title(f"{run_id} — {label}")

            if ax is axes[0]:
                ax.set_ylabel(f"Residual pred - true [{unit}]")

            ax.legend()

        fig.suptitle(
            f"Residual density vs true value — {label} — split={split}",
            y=1.02,
        )
        plt.tight_layout()
        plt.show()


plot_residual_density_by_true(
    run_ids=("M08", "M09"),
    split="test",
    physical_space=True,
    gridsize=85,
    trend_bins=35,
)

## 17. Compact true-vs-predicted comparison tables

These tables quantify the visual pattern shown above: slope, dispersion ratio, and high-percentile absolute error.

In [ ]:
true_pred_summary_cols = [
    "label",
    "slope_M08",
    "slope_M09",
    "slope_delta",
    "std_ratio_M08",
    "std_ratio_M09",
    "std_ratio_delta",
]

display(slope_delta_df[true_pred_summary_cols])

tail_summary = (
    test_quantile_df
    .query("quantile in [0.90, 0.95, 0.99]")
    .copy()
)

display(tail_summary)

## 18. Final decision

### Training behavior

M08 and M09 reach their best validation checkpoints at the same epoch, epoch 36, and both stop at epoch 61. M09 obtains a lower best validation loss:

- M08 best validation loss: 0.116116
- M09 best validation loss: 0.114245

This is a relative validation improvement of approximately 1.61%. However, M09 also ends with a lower final training loss than M08 while having a similar final validation loss. This suggests that the attention-pooling model has slightly higher effective capacity, but the validation gain is modest.

### Test performance

On the standardized test set, M09 improves the global metrics relative to M08:

- global MSE: 0.116352 → 0.113651, a 2.32% reduction;
- global MAE: 0.244785 → 0.241269, a 1.44% reduction.

The improvement is present for all three targets:

- chirp mass: MSE −1.35%, MAE −1.53%;
- total mass: MSE −3.69%, MAE −2.51%;
- effective spin: MSE −2.26%, MAE −0.71%.

The paired bootstrap intervals are below zero for all target-level MSE and MAE deltas, so the test-set improvement is statistically detectable under fixed trained models. The strongest target-level improvement is total mass.

However, the gain is small in practical terms. It is also smaller than the seed-to-seed variability observed for M08, where seed124 improved the global MSE by about 3.6% relative to M08 seed123. Therefore, M09 cannot yet be considered a clearly superior architecture from this single seed.

### Regression and contraction

For the mass parameters, M09 slightly improves the regression slopes and prediction dispersion:

- chirp-mass slope: 0.8728 → 0.8749;
- total-mass slope: 0.8934 → 0.8994.

This is favorable but small.

For effective spin, M09 reduces the slope and dispersion:

- chi_eff slope: 0.8118 → 0.7800;
- std(pred)/std(true): 0.8983 → 0.8597.

This is the main warning sign. M09 reduces the chi_eff MSE, but it does so while producing a more compressed prediction distribution. The MAE gain is also small for chi_eff, and the q99 absolute-error quantile slightly worsens. This suggests that M09 may be smoothing or regularizing the effective-spin predictions rather than learning a substantially richer temporal representation.

### Error tails

M09 improves most physical absolute-error quantiles, but the tail improvements are modest:

- chirp mass q95: −0.64%, q99: −0.37%;
- total mass q95: −1.19%, q99: −0.33%;
- chi_eff q95: −1.65%, q99: +0.33%.

Thus, M09 does not produce a strong reduction of difficult-event tails. The q99 degradation for chi_eff is small, but it reinforces the point that the attention pooling is not obviously solving the hardest effective-spin cases.

### Attention-window diagnostic

The attention windows are not completely trivial. The four windows often concentrate on different encoder-time regions, and at least one window frequently forms a sharper peak that may plausibly correspond to the high-information part of the signal, such as the merger/ringdown region.

This is useful diagnostically: the model is using the attention mechanism in a non-uniform way. However, the metric gains indicate that this extra temporal selection is not translating into a large improvement in parameter estimation. A plausible interpretation is that the residual dilated encoder already extracts most of the relevant temporal information before pooling, so replacing average pooling with multi-slot attention only gives a small refinement.

### Adoption decision

M09 should not replace M08 as the operational baseline yet.

The result is best interpreted as a positive but marginal pooling ablation:

1. M09 improves global test MSE and MAE relative to M08 seed123.
2. The improvement is consistent across targets and statistically detectable on the fixed test set.
3. The practical gain is small.
4. The gain is smaller than the previously observed M08 seed-to-seed variability.
5. M09 worsens the effective-spin slope and prediction dispersion.
6. The high-error tails improve only weakly and chi_eff q99 slightly worsens.
7. The attention windows are interpretable, but they do not provide evidence of a major new performance regime.

Therefore, M08 remains the current operational baseline. M09 should be retained as an informative ablation showing that learned temporal pooling can provide a small improvement, especially for total mass, but not enough to justify replacing M08 without additional evidence.

### Recommended next steps

The next priority should not be another pooling variant immediately. The evidence suggests that pooling is not the dominant bottleneck once the residual dilated encoder is in place.

Recommended sequence:

1. Keep M08 as the baseline for the main Mondrian/conformal analysis.
2. Optionally evaluate M09 with Mondrian only after the M08/M00 conformal story is finalized.
3. If M09 is pursued further, run a second seed before making architectural claims.
4. Inspect attention windows in relation to physical quantities: merger location, mass, signal duration, SNR, and residual size.
5. If searching for a larger architecture gain, test a deeper non-causal TCN or a richer residual-dilated encoder rather than more pooling-only variants.

The current conclusion is that M09 is scientifically useful, but not operationally decisive.


## 19. Attention windows and GW-region metadata

This section is intentionally conservative.

The event-level HDF5 metadata contains several coordinate systems:

- `injection/<det>/overlap_*_index_signal`: indices in the generated signal array.
- `injection/<det>/overlap_*_index_strain`: indices in the strain array at the injection stage.
- `windowing/*`: absolute GPS times and durations used by the projection/window selection step.
- `X[event, :, :]`: the final model input after processing/cropping.

The figures below therefore compare multiple candidate mappings instead of assuming that one metadata field is automatically in final-`X` coordinates.

If candidate spans disagree with the attention windows, do **not** interpret that immediately as a model failure. It may simply mean that the metadata span is expressed before the final crop used to build `X`.


In [ ]:
# ============================================================
# HDF5 metadata discovery and ordered reads
# ============================================================

import h5py
import json
from pathlib import Path

dataset_path = PROCESSED_ROOT / f"{DATASET_ID}.h5"
if not dataset_path.exists():
    dataset_path = PROCESSED_ROOT / DATASET_ID / f"{DATASET_ID}.h5"

assert dataset_path.exists(), dataset_path

print("dataset_path:", dataset_path)

def discover_hdf5_datasets(h5_path):
    rows = []
    with h5py.File(h5_path, "r") as f:
        n_samples = f["X"].shape[0]

        def visitor(name, obj):
            if isinstance(obj, h5py.Dataset):
                shape = tuple(obj.shape)
                event_level = len(shape) > 0 and shape[0] == n_samples
                rows.append({
                    "path": name,
                    "shape": shape,
                    "dtype": str(obj.dtype),
                    "event_level": event_level,
                    "ndim": len(shape),
                })

        f.visititems(visitor)

    return pd.DataFrame(rows).sort_values("path").reset_index(drop=True)


metadata_fields_df = discover_hdf5_datasets(dataset_path)

useful_event_fields_df = metadata_fields_df[
    metadata_fields_df["event_level"]
    & metadata_fields_df["path"].str.contains(
        "injection|placement|projection|windowing|snr|parameters",
        regex=True,
    )
].copy()

display(useful_event_fields_df)


def read_hdf5_dataset_preserve_order(
    h5_file,
    dataset_key,
    indices,
    dtype=None,
):
    """
    h5py requires fancy indices to be increasing. This helper reads sorted
    indices and restores the original requested order.
    """
    indices = np.asarray(indices, dtype=np.int64)

    order = np.argsort(indices, kind="stable")
    sorted_indices = indices[order]

    data_sorted = np.asarray(
        h5_file[dataset_key][sorted_indices],
        dtype=dtype,
    )

    inverse_order = np.empty_like(order)
    inverse_order[order] = np.arange(len(order))

    return data_sorted[inverse_order]


def read_event_metadata(
    h5_path,
    hdf5_indices,
    field_paths=None,
    max_vector_width=8,
):
    hdf5_indices = np.asarray(hdf5_indices, dtype=np.int64)
    columns = {"hdf5_index": hdf5_indices}

    with h5py.File(h5_path, "r") as f:
        n_samples = f["X"].shape[0]

        if field_paths is None:
            field_paths = []
            for _, meta_row in metadata_fields_df.iterrows():
                path = meta_row["path"]
                shape = tuple(meta_row["shape"])
                event_level = bool(meta_row["event_level"])
                ndim = int(meta_row["ndim"])

                if not event_level:
                    continue

                if ndim == 1:
                    field_paths.append(path)
                elif ndim == 2 and len(shape) > 1 and shape[1] <= max_vector_width:
                    field_paths.append(path)

        for path in field_paths:
            if path not in f:
                continue

            obj = f[path]
            if len(obj.shape) == 0 or obj.shape[0] != n_samples:
                continue

            values = read_hdf5_dataset_preserve_order(
                h5_file=f,
                dataset_key=path,
                indices=hdf5_indices,
            )

            if values.ndim == 1:
                columns[path] = values
            elif values.ndim == 2 and values.shape[1] <= max_vector_width:
                for j in range(values.shape[1]):
                    columns[f"{path}[{j}]"] = values[:, j]

    return pd.DataFrame(columns)


def find_metadata_jsons():
    candidates = []
    roots = [
        PROCESSED_ROOT,
        PROCESSED_ROOT / DATASET_ID,
        DATA_ROOT,
    ]
    names = [
        f"{DATASET_ID}.metadata.json",
        f"{DATASET_ID}_splits_train400000_val40000_cal30000_test30000_seed123.metadata.json",
        f"{DATASET_ID}_splits_train400000_val40000_cal30000_test30000_seed123.metadata",
    ]

    for root in roots:
        if not root.exists():
            continue
        for name in names:
            p = root / name
            if p.exists():
                candidates.append(p)
        candidates.extend(root.glob(f"{DATASET_ID}*metadata*.json"))
        candidates.extend(root.glob(f"{DATASET_ID}*metadata"))

    return sorted(set(candidates))


metadata_json_candidates = find_metadata_jsons()
print("\nMetadata JSON candidates:")
for p in metadata_json_candidates:
    print(" ", p)


## 20. Candidate GW-region mappings

The main diagnostic issue is that the injected-signal metadata may not be in the same coordinate frame as the final model input `X`.

This cell constructs several candidate spans:

- `raw_overlap_strain`: direct use of `injection/<det>/overlap_*_index_strain`.
- `shifted_by_used_window_start`: subtracts the offset between `windowing/used_window_start_time` and the detector segment start time.
- `windowing_used_duration_at_start`: maps the selected network window to the beginning of final `X`.
- `windowing_full_network_from_segment_time`: uses `windowing/full_network_*_time` relative to detector segment start time.

Only a candidate that is consistent with the actual dataset construction should be interpreted as the real GW region. This notebook does not force a single interpretation when the candidates disagree.


In [ ]:
# ============================================================
# Candidate GW-region inference
# ============================================================

DETECTORS = ["H1", "L1", "V1"]
FS = 4096.0

def _float_or_nan(value):
    try:
        return float(value)
    except Exception:
        return np.nan


def _valid_span(start, end, signal_length):
    start = _float_or_nan(start)
    end = _float_or_nan(end)

    if not np.isfinite(start) or not np.isfinite(end):
        return False
    if end <= start:
        return False
    if end < 0 or start > signal_length:
        return False
    return True


def _clip_span(start, end, signal_length):
    start = int(np.floor(max(0, start)))
    end = int(np.ceil(min(signal_length - 1, end)))
    return start, end


def _union_spans(spans, source):
    spans = [s for s in spans if s is not None]
    if not spans:
        return None

    return {
        "source": source,
        "start": min(s["start"] for s in spans),
        "end": max(s["end"] for s in spans),
        "detector_spans": spans,
    }


def get_detector_segment_start(row, detector):
    for key in [
        f"injection/{detector}/segment_start_time",
        f"projection/{detector}/segment_start_time",
        "placement/segment_start_time",
    ]:
        if key in row.index and pd.notna(row[key]):
            return _float_or_nan(row[key])
    return np.nan


def candidate_raw_overlap_strain(row, signal_length):
    spans = []

    for det in DETECTORS:
        sk = f"injection/{det}/overlap_start_index_strain"
        ek = f"injection/{det}/overlap_end_index_strain"

        if sk not in row.index or ek not in row.index:
            continue

        start = _float_or_nan(row[sk])
        end = _float_or_nan(row[ek])

        if _valid_span(start, end, signal_length):
            start, end = _clip_span(start, end, signal_length)
            spans.append({
                "detector": det,
                "start": start,
                "end": end,
            })

    return _union_spans(spans, "raw_overlap_strain")


def candidate_shifted_by_used_window_start(row, signal_length):
    spans = []

    if "windowing/used_window_start_time" not in row.index:
        return None

    used_start_time = _float_or_nan(row["windowing/used_window_start_time"])

    for det in DETECTORS:
        sk = f"injection/{det}/overlap_start_index_strain"
        ek = f"injection/{det}/overlap_end_index_strain"

        if sk not in row.index or ek not in row.index:
            continue

        seg_start = get_detector_segment_start(row, det)
        if not np.isfinite(seg_start) or not np.isfinite(used_start_time):
            continue

        crop_offset = round((used_start_time - seg_start) * FS)

        start = _float_or_nan(row[sk]) - crop_offset
        end = _float_or_nan(row[ek]) - crop_offset

        if _valid_span(start, end, signal_length):
            start, end = _clip_span(start, end, signal_length)
            spans.append({
                "detector": det,
                "start": start,
                "end": end,
                "crop_offset": crop_offset,
            })

    return _union_spans(spans, "shifted_by_used_window_start")


def candidate_windowing_used_duration_at_start(row, signal_length):
    if "windowing/used_window_duration" not in row.index:
        return None

    duration = _float_or_nan(row["windowing/used_window_duration"])
    if not np.isfinite(duration):
        return None

    width = int(round(duration * FS))
    start = 0
    end = width

    if _valid_span(start, end, signal_length):
        start, end = _clip_span(start, end, signal_length)
        return {
            "source": "windowing_used_duration_at_start",
            "start": start,
            "end": end,
            "detector_spans": [],
        }

    return None


def candidate_full_network_from_segment_time(row, signal_length):
    if (
        "windowing/full_network_start_time" not in row.index
        or "windowing/full_network_end_time" not in row.index
    ):
        return None

    full_start = _float_or_nan(row["windowing/full_network_start_time"])
    full_end = _float_or_nan(row["windowing/full_network_end_time"])

    spans = []

    for det in DETECTORS:
        seg_start = get_detector_segment_start(row, det)
        if not np.isfinite(seg_start):
            continue

        start = (full_start - seg_start) * FS
        end = (full_end - seg_start) * FS

        if _valid_span(start, end, signal_length):
            start, end = _clip_span(start, end, signal_length)
            spans.append({
                "detector": det,
                "start": start,
                "end": end,
            })

    return _union_spans(spans, "windowing_full_network_from_segment_time")


def infer_candidate_regions(row, signal_length):
    candidates = []

    for fn in [
        candidate_raw_overlap_strain,
        candidate_shifted_by_used_window_start,
        candidate_windowing_used_duration_at_start,
        candidate_full_network_from_segment_time,
    ]:
        out = fn(row, signal_length)
        if out is not None:
            out["width"] = out["end"] - out["start"]
            candidates.append(out)

    return candidates


def candidate_summary_for_events(event_metadata_table, signal_length=16384):
    rows = []

    for _, row in event_metadata_table.iterrows():
        candidates = infer_candidate_regions(row, signal_length)

        for cand in candidates:
            rows.append({
                "hdf5_index": int(row["hdf5_index"]),
                "source": cand["source"],
                "start": cand["start"],
                "end": cand["end"],
                "width": cand["width"],
            })

    return pd.DataFrame(rows)


## 21. Plot attention windows with candidate GW regions

The plot below overlays attention windows with all plausible GW-region mappings. Use this as a coordinate-frame diagnostic, not as definitive proof that a window is or is not attending to the signal.

If only one candidate is physically consistent with the generator implementation, keep that candidate in the final report and remove the others. If they disagree and the generator code is not immediately checked, keep the conclusion conservative.


In [ ]:
# ============================================================
# Attention windows + candidate GW-region overlays
# ============================================================

import torch
from src.models.network import SimpleCNN_ResidualDilatedMultiAttention

def load_m09_model_from_checkpoint():
    checkpoint = torch.load(
        RUNS["M09"]["checkpoint_path"],
        map_location="cpu",
    )

    model_config = checkpoint.get("model_config", {})
    model_kwargs = model_config.get("model_kwargs", {})

    model_kwargs.setdefault("n_detectors", 3)
    model_kwargs.setdefault("n_outputs", 3)
    model_kwargs.setdefault("embedding_dim", 64)
    model_kwargs.setdefault("residual_channels", 64)
    model_kwargs.setdefault("dilations", [1, 2, 4])
    model_kwargs.setdefault("residual_kernel_size", 7)
    model_kwargs.setdefault("dropout_conv", 0.05)
    model_kwargs.setdefault("dropout_dense", 0.1)
    model_kwargs.setdefault("num_groups", 8)
    model_kwargs.setdefault("attention_slots", 4)
    model_kwargs.setdefault("attention_hidden", 64)
    model_kwargs.setdefault("attention_dropout", 0.0)
    model_kwargs.setdefault("slot_dim", 32)

    model = SimpleCNN_ResidualDilatedMultiAttention(**model_kwargs)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    return model


def attention_to_signal_grid(attention_weights, signal_length):
    n_encoder = attention_weights.shape[-1]
    return np.linspace(0, signal_length - 1, n_encoder)


def event_title_from_metadata(row, y_true_phys=None, err_info=None):
    fields = []

    for key, label in [
        ("parameters/chirp_mass", "Mc"),
        ("parameters/total_mass", "Mtot"),
        ("parameters/chi_eff", "chi_eff"),
        ("snr/network", "SNR"),
        ("snr/target_network", "target SNR"),
    ]:
        if key in row.index and pd.notna(row[key]):
            fields.append(f"{label}={float(row[key]):.3g}")

    if y_true_phys is not None and not fields:
        fields.extend([
            f"Mc={y_true_phys[0]:.3g}",
            f"Mtot={y_true_phys[1]:.3g}",
            f"chi_eff={y_true_phys[2]:.3g}",
        ])

    if err_info is not None:
        fields.append(
            f"mean |err| std: M08={err_info['m08']:.3f}, M09={err_info['m09']:.3f}"
        )

    return " | ".join(fields)


def plot_attention_windows_with_candidate_regions(
    sample_positions,
    split="test",
    run_id="M09",
    max_samples=8,
    normalize_signal=True,
):
    dataset_path = PROCESSED_ROOT / f"{DATASET_ID}.h5"
    if not dataset_path.exists():
        dataset_path = PROCESSED_ROOT / DATASET_ID / f"{DATASET_ID}.h5"

    assert dataset_path.exists(), dataset_path

    model = load_m09_model_from_checkpoint()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    idx_split = ALIGNED[split][run_id]["idx"]

    sample_positions = np.asarray(list(sample_positions)[:max_samples], dtype=int)
    hdf5_indices = idx_split[sample_positions]

    with h5py.File(dataset_path, "r") as f:
        X = read_hdf5_dataset_preserve_order(
            h5_file=f,
            dataset_key="X",
            indices=hdf5_indices,
            dtype=np.float32,
        )

    event_metadata_table = read_event_metadata(
        dataset_path,
        hdf5_indices=hdf5_indices,
    )

    metadata_by_hdf5 = {
        int(row["hdf5_index"]): row
        for _, row in event_metadata_table.iterrows()
    }

    x_tensor = torch.from_numpy(X).to(device)

    with torch.no_grad():
        features, attention = model.get_attention_weights(x_tensor)

    attention = attention.detach().cpu().numpy()

    y_true_phys_all = inverse_standardize(ALIGNED[split][run_id]["y"])
    ae_m08_std = np.abs(ALIGNED[split]["M08"]["pred"] - ALIGNED[split][run_id]["y"])
    ae_m09_std = np.abs(ALIGNED[split]["M09"]["pred"] - ALIGNED[split][run_id]["y"])

    for local_i, hdf5_idx in enumerate(hdf5_indices):
        strain = X[local_i]
        n_detectors, signal_length = strain.shape

        energy = np.sqrt(np.mean(strain**2, axis=0))
        if normalize_signal:
            max_energy = np.max(np.abs(energy))
            energy_plot = energy / max_energy if max_energy > 0 else energy
        else:
            energy_plot = energy

        row = metadata_by_hdf5[int(hdf5_idx)]
        candidates = infer_candidate_regions(row, signal_length)

        attn = attention[local_i]
        original_grid = attention_to_signal_grid(attn, signal_length)

        attention_peak_samples = [
            float(original_grid[np.argmax(attn[k])])
            for k in range(attn.shape[0])
        ]

        split_position = sample_positions[local_i]

        title = event_title_from_metadata(
            row,
            y_true_phys=y_true_phys_all[split_position],
            err_info={
                "m08": float(ae_m08_std[split_position].mean()),
                "m09": float(ae_m09_std[split_position].mean()),
            },
        )

        print(
            f"HDF5 idx={int(hdf5_idx)} | attention peak samples="
            f"{[round(v, 1) for v in attention_peak_samples]}"
        )
        for cand in candidates:
            print(
                f"  candidate {cand['source']}: "
                f"{cand['start']}–{cand['end']} width={cand['width']}"
            )

        fig, ax1 = plt.subplots(figsize=(14.0, 5.2))

        ax1.plot(
            np.arange(signal_length),
            energy_plot,
            linewidth=1.0,
            label="multidetector strain energy",
        )

        # Plot candidate regions as outlines/faint spans.
        # Different line styles reduce reliance on color.
        region_styles = {
            "raw_overlap_strain": {"alpha": 0.13, "hatch": None},
            "shifted_by_used_window_start": {"alpha": 0.18, "hatch": "//"},
            "windowing_used_duration_at_start": {"alpha": 0.10, "hatch": "\\\\"},
            "windowing_full_network_from_segment_time": {"alpha": 0.08, "hatch": ".."},
        }

        for cand in candidates:
            style = region_styles.get(cand["source"], {"alpha": 0.10, "hatch": None})
            ax1.axvspan(
                cand["start"],
                cand["end"],
                alpha=style["alpha"],
                hatch=style["hatch"],
                label=f"candidate GW region: {cand['source']}",
            )

        ax1.set_xlabel("Original sample index in final X window")
        ax1.set_ylabel("Normalized strain energy")
        ax1.set_title(
            f"HDF5 idx={int(hdf5_idx)} | {title}"
        )

        ax2 = ax1.twinx()

        for k in range(attn.shape[0]):
            weights = attn[k]
            w = weights / np.max(weights) if np.max(weights) > 0 else weights

            ax2.plot(
                original_grid,
                w,
                linewidth=1.5,
                alpha=0.90,
                label=f"attention window {k} | peak={attention_peak_samples[k]:.0f}",
            )

        ax2.set_ylabel("Attention weight, normalized per window")

        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()

        # Avoid an unreadably large legend if candidates overlap.
        ax1.legend(
            lines1 + lines2,
            labels1 + labels2,
            loc="upper right",
            fontsize=8,
        )

        plt.tight_layout()
        plt.show()


# Default examples: random test samples.
rng = np.random.default_rng(123)
sample_positions = rng.choice(
    len(ALIGNED["test"]["M09"]["idx"]),
    size=6,
    replace=False,
)

plot_attention_windows_with_candidate_regions(
    sample_positions=sample_positions,
    split="test",
    run_id="M09",
    max_samples=6,
)


## 22. Select interpretable events for attention-window inspection

This repeats the same plotting logic for events where M09 helps, events where M09 hurts, and the lowest/highest-mass systems. These examples are better for interpretation than random draws.


In [ ]:
# ============================================================
# Select interpretable events for attention-window inspection
# ============================================================

def build_event_selection_table(split="test"):
    y_true_std = ALIGNED[split]["M09"]["y"]
    pred_m08_std = ALIGNED[split]["M08"]["pred"]
    pred_m09_std = ALIGNED[split]["M09"]["pred"]

    y_true_phys = inverse_standardize(y_true_std)

    ae_m08_std = np.abs(pred_m08_std - y_true_std)
    ae_m09_std = np.abs(pred_m09_std - y_true_std)

    rows = []

    for i, hdf5_idx in enumerate(ALIGNED[split]["M09"]["idx"]):
        rows.append({
            "position": i,
            "hdf5_index": int(hdf5_idx),
            "chirp_mass": y_true_phys[i, 0],
            "total_mass": y_true_phys[i, 1],
            "chi_eff": y_true_phys[i, 2],
            "ae_m08_mean_std": ae_m08_std[i].mean(),
            "ae_m09_mean_std": ae_m09_std[i].mean(),
            "m09_minus_m08_ae_mean_std": ae_m09_std[i].mean() - ae_m08_std[i].mean(),
            "ae_m08_chirp_std": ae_m08_std[i, 0],
            "ae_m09_chirp_std": ae_m09_std[i, 0],
            "ae_m08_total_std": ae_m08_std[i, 1],
            "ae_m09_total_std": ae_m09_std[i, 1],
            "ae_m08_chi_std": ae_m08_std[i, 2],
            "ae_m09_chi_std": ae_m09_std[i, 2],
        })

    df = pd.DataFrame(rows)

    selected = []

    selected.append(
        df.nsmallest(3, "m09_minus_m08_ae_mean_std")
        .assign(selection_reason="M09 helps most")
    )

    selected.append(
        df.nlargest(3, "m09_minus_m08_ae_mean_std")
        .assign(selection_reason="M09 hurts most")
    )

    selected.append(
        df.nsmallest(3, "total_mass")
        .assign(selection_reason="lowest total mass / longest signals")
    )

    selected.append(
        df.nlargest(3, "total_mass")
        .assign(selection_reason="highest total mass / shortest signals")
    )

    selected.append(
        df.nlargest(3, "ae_m09_chi_std")
        .assign(selection_reason="largest chi_eff error in M09")
    )

    out = pd.concat(selected, ignore_index=True)
    out = out.drop_duplicates(subset=["position"]).reset_index(drop=True)

    return out


event_selection_df = build_event_selection_table(split="test")
display(event_selection_df)

plot_attention_windows_with_candidate_regions(
    sample_positions=event_selection_df["position"].values,
    split="test",
    run_id="M09",
    max_samples=len(event_selection_df),
)


## 23. Interpretation of attention-window overlays

The current evidence should be interpreted conservatively.

The images show that the four attention windows are not uniform and often form sharp, event-dependent peaks. Therefore, M09 is using its attention mechanism rather than collapsing to average pooling.

However, the GW-region overlay is sensitive to coordinate-frame interpretation. In particular, `injection/<det>/overlap_*_index_strain` is not guaranteed to be in the final `X[event, :, :]` coordinate frame after network windowing and cropping. If the candidate GW regions disagree, the notebook should not claim that the windows are synchronized with the injected signal.

Practical conclusion:

```text
M08 remains the operational baseline.
M09 is a positive but marginal pooling ablation.
The attention windows are potentially interpretable, but their physical alignment with the injected GW signal requires checking the dataset generation coordinate convention.
```

For the final report, use these plots only as an exploratory diagnostic unless the generator code confirms which metadata field maps exactly to final `X` coordinates.
